In [ ]:
import os
import json
import re
import math
import numpy as np
import random
import pandas as pd
import tellurium as te
import logging
from datetime import datetime
from typing import Dict

In [ ]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

In [ ]:
# Code to generate 3x3 quantity/quality synthetic datasets by simulating a ground-truth model
# Save as synth_experiments.py and run in the environment where Tellurium is available.
# Usage:
#   - set MODEL_ANTIMONY string below (or provide SBML path and set USE_SBML=True)
#   - run: python synth_experiments.py
#
# The script:
#  - generates 100 high-quantity starting conditions sampling chosen metabolites on a discrete grid
#  - produces medium (50 random subset) and low (10 random subset) sets
#  - simulates ground truth trajectories for each starting condition
#  - samples concentrations at t = [0,1,2,4] hours
#  - generates noisy observed datasets for quality levels: high (1% rel noise), medium (5%), low (15%)
#  - saves CSVs and a metadata JSON


##### USER CONFIGURATION #####
# Provide either an Antimony string (ANTIMONY_MODEL) or an SBML file path (SBML_PATH).
# If both are provided, ANTIMONY_MODEL takes precedence.
ANTIMONY_MODEL = r'iMC057.txt'  # e.g., """model mymod; A -> B; A = 1; end"""

# Name for output directory
OUTDIR = "synthetic_datasets"

# Metabolites to vary and their ranges (min, max) inclusive. Step is 0.5 per your spec.
VARY_METABOLITES = {
    "ATP": (0, 2, 0.1),
    "AcetylCoA": (0, 2, 0.1),
    "L_Serine": (0, 2, 0.1),
    "NADH": (0, 2, 0.1),
    "AMP": (0, 2, 0.1),
    "HCO3": (0, 10, 0.5),
    "Hydroxycitrate": (0, 10, 0.5),
    "NAD": (0, 2, 0.1),
    "NADPH": (0, 2, 0.1),
    "NADP": (0, 2, 0.1),
    "Pyruvate": (0, 2, 0.1),
    "SMalate": (0, 2, 0.1),   # fixed point (only 2.0 will be chosen)
    "ADP": (0, 2, 0.1),
    "CoA": (0, 2, 0.1),
    "Oxaloacetate": (0, 2, 0.1),
    "Formate": (0, 2, 0.1),
    "hEC6411": (0, 0.05, 0.005),
    "hEC43117": (0, 0.05, 0.005),
    "eEC11137": (0.01, 0.05, 0.001)
}

# Number of experiments for high/medium/low quantity
N_HIGH = 100
N_MED = 50    # will be sampled from high set
N_LOW = 25    # will be sampled from medium set

# Timepoints (hours) to sample from simulations (convert to same time units as model)
SAMPLE_TIMES_HOURS = [0, 1, 2, 4]  # hours
# If your model uses seconds/minutes, set TIME_UNITS_PER_HOUR appropriately (default assumes hours)
TIME_UNITS_PER_HOUR = 1.0  # set to 3600 if model uses seconds, 60 if minutes, 1 if hours

# Quality noise levels (relative gaussian noise std dev)
QUALITY_NOISE = {
    "high": 0.1,    # 10% relative noise
    "medium": 0.25,  # 25%
    "low": 0.50      # 50%
}

# Random seed for reproducibility
SEED = 12345
np.random.seed(SEED)
#############################

In [ ]:
def load_model():
    """Load the ground truth model into a tellurium RoadRunner object and return it.
    Also return a dict of default initial concentrations {species_id: value}."""
    if ANTIMONY_MODEL:
        rr = te.loada(ANTIMONY_MODEL)
    else:
        raise ValueError("Please set ANTIMONY_MODEL (preferred) or SBML_PATH above to load the ground truth model.")

    # Collect species ids and their initial concentrations
    species_ids = list(rr.getFloatingSpeciesIds())
    initial_vals = {sid: float(rr[sid]) for sid in species_ids}
    logging.info("Loaded model with %d floating species.", len(species_ids))
    return rr, initial_vals


def discrete_sample(min_val, max_val, step=0.5, size=1):
    """Sample uniformly at random from discrete grid with given step."""
    grid = np.arange(min_val, max_val + 1e-9, step)
    return np.random.choice(grid, size=size)


def generate_high_starting_conditions(initial_vals, vary_mets, n_high):
    """Generate n_high starting condition dicts by sampling only the vary_mets on discrete grid,
    while other species stay as in initial_vals."""
    starts = []
    grid_choices = {}
    for met, (mn, mx, step) in vary_mets.items():
        grid_choices[met] = np.arange(mn, mx + 1e-9, step)

    for i in range(n_high):
        sc = dict(initial_vals)  # copy defaults
        for met, grid in grid_choices.items():
            if met not in sc:
                raise KeyError(f"Metabolite '{met}' not found among model species. "
                               "Species IDs must match exactly.")
            sc[met] = float(np.random.choice(grid))
        sc['_exp_id'] = f"exp_{i+1:03d}"
        starts.append(sc)
    return starts


def simulate_start(rr, initial_cond, times_hours):
    """Simulate the model with a given initial_cond dict and return a pandas DataFrame of results.
    times_hours are in hours; converted to model time units via TIME_UNITS_PER_HOUR."""
    # set species initial values
    for sid, val in initial_cond.items():
        if sid == '_exp_id':
            continue
        # assign to model (this sets the current value; for a fresh simulation, reset to initial step)
        rr[sid] = float(val)

    # set simulation times
    times = [t * TIME_UNITS_PER_HOUR for t in times_hours]
    # ensure we have at least one point at t=0
    sim = rr.simulate(times[0], times[-1], int( (times[-1]-times[0]) / max(1e-9, (times[1]-times[0])) ) + 1 )
    # sim is array with columns time + species

    # Convert to DataFrame and extract rows at the requested times (use nearest)
    if isinstance(sim, np.ndarray):
        colnames = ["time"] + rr.getFloatingSpeciesIds()
        # In case boundary species are also included, append them too
        if len(colnames) < sim.shape[1]:
            colnames += rr.getBoundarySpeciesIds()
        df = pd.DataFrame(sim, columns=colnames)
    else:
        df = pd.DataFrame(sim)

    # df = pd.DataFrame(sim, columns=rr.getFloatingSpeciesIds()[:len(sim[0])]) if isinstance(sim, np.ndarray) else pd.DataFrame(sim)
    # Tellurium's simulate returns numpy array; first column is time named 'time' in older versions — use rr.getSimulationResult()
    try:
        sim_res = rr.getSimulationResult()
        times_arr = np.array(sim_res['time'])
        colnames = sim_res.colnames  # ['time','A','B',...']
        df = pd.DataFrame(sim_res, columns=colnames)
    except Exception:
        # fallback: try building from sim and rr species ids
        colnames = ['time'] + list(rr.getFloatingSpeciesIds())
        df = pd.DataFrame(sim, columns=colnames)

    # pick timepoints by nearest match
    out_rows = []
    for t in times:
        # find index of closest time in df['time']
        idx = (np.abs(df['time'] - t)).idxmin()
        row = df.loc[idx, :].to_dict()
        out_rows.append({'time': t, **row})

    return pd.DataFrame(out_rows)


def add_noise_to_obs(df_gt, measured_species, quality_level):
    """Given ground truth df with columns time and species, sample measured timepoints and add noise.
    Returns DataFrame in long format: exp_id, time_hr, species, observed_conc."""
    sigma_rel = QUALITY_NOISE[quality_level]
    rows = []
    for _, row in df_gt.iterrows():
        time_hr = row['time'] / TIME_UNITS_PER_HOUR
        for sp in measured_species:
            if sp not in row:
                raise KeyError(f"Measured species '{sp}' not found in simulation result columns.")
            true_val = float(row[sp])
            # relative gaussian noise, ensure non-negative
            obs = true_val * (1 + np.random.normal(0, sigma_rel))
            if obs < 0:
                obs = 0.0
            rows.append({
                "time_hr": time_hr,
                "species": sp,
                "true_value": true_val,
                "observed": obs
            })
    return pd.DataFrame(rows)


def main():
    os.makedirs(OUTDIR, exist_ok=True)
    rr, default_initials = load_model()

    # Validate that all vary metabolites and measured species exist in model floats
    model_species = set(default_initials.keys())

    for met in VARY_METABOLITES.keys():
        if met not in model_species:
            raise KeyError(f"Vary metabolite '{met}' not found in model species. Available species: {sorted(model_species)[:20]}")

    # List of key measured species (customize as needed)
    measured_species = ["SMalate", "ATP", "Pyruvate", "NADH", "ADP", "AMP", "AcetylCoA", "Citrate", "CoA", "Fumarate", "Glycine", "Glyoxylate", "L_Serine", "NAD", "NADPH", "NADP", "Phosphoenolpyruvate", "Succinate", "n2Oxoglutarate", "Formate"]  # adjust to your actual model species IDs
    for sp in measured_species:
        if sp not in model_species:
            raise KeyError(f"Measured species '{sp}' not found in model species list.")

    # Generate high starting conditions
    high_starts = generate_high_starting_conditions(default_initials, VARY_METABOLITES, N_HIGH)
    # Sample medium and low as subsets per your instructions
    medium_indices = np.random.choice(range(N_HIGH), size=N_MED, replace=False)
    medium_starts = [high_starts[i] for i in medium_indices]
    low_indices = np.random.choice(range(N_MED), size=N_LOW, replace=False)
    low_starts = [medium_starts[i] for i in low_indices]

    # persist metadata
    meta = {
        "generated_at": datetime.utcnow().isoformat() + "Z",
        "seed": int(SEED),
        "vary_metabolites": VARY_METABOLITES,
        "n_high": N_HIGH, "n_med": N_MED, "n_low": N_LOW,
        "measured_species": measured_species,
        "times_hr": SAMPLE_TIMES_HOURS,
        "quality_noise": QUALITY_NOISE
    }
    with open(os.path.join(OUTDIR, "metadata.json"), "w") as f:
        json.dump(meta, f, indent=2)

    # Helper to run a set and save
    def run_and_save(starting_conditions, label_prefix):
        gt_records = []   # ground truth summary (one row per exp per time capture)
        obs_records = {q: [] for q in QUALITY_NOISE.keys()}  # dictionaries of lists
        for sc in starting_conditions:
            exp_id = sc['_exp_id']
            logging.info("Simulating %s (label %s)", exp_id, label_prefix)
            df_gt = simulate_start(rr, sc, SAMPLE_TIMES_HOURS)
            # insert exp_id
            df_gt.insert(0, "exp_id", exp_id)
            # save ground truth time-series per experiment
            gt_records.append(df_gt.assign(exp_id=exp_id))
            # build observed noisy datasets for each quality
            for q in QUALITY_NOISE.keys():
                obs = add_noise_to_obs(df_gt, measured_species, q)
                obs.insert(0, "exp_id", exp_id)
                obs_records[q].append(obs)

        # concatenate and save
        gt_df = pd.concat(gt_records, ignore_index=True)
        gt_df.to_csv(os.path.join(OUTDIR, f"{label_prefix}_ground_truth_timeseries.csv"), index=False)

        for q, recs in obs_records.items():
            if recs:
                obs_df = pd.concat(recs, ignore_index=True)
                obs_df.to_csv(os.path.join(OUTDIR, f"{label_prefix}_observations_{q}.csv"), index=False)

        # also save starting conditions used (flatten dicts)
        starts_flat = []
        for sc in starting_conditions:
            sc_flat = {k: v for k, v in sc.items() if k != '_exp_id'}
            sc_flat['exp_id'] = sc['_exp_id']
            starts_flat.append(sc_flat)
        pd.DataFrame(starts_flat).to_csv(os.path.join(OUTDIR, f"{label_prefix}_starting_conditions.csv"), index=False)
        return gt_df

    # Run high, medium, low and save
    logging.info("Running high quantity (%d) experiments", N_HIGH)
    gt_high = run_and_save(high_starts, "highQ")
    logging.info("Running medium quantity (%d) experiments", N_MED)
    gt_med = run_and_save(medium_starts, "medQ")
    logging.info("Running low quantity (%d) experiments", N_LOW)
    gt_low = run_and_save(low_starts, "lowQ")

    logging.info("All datasets generated in %s", OUTDIR)

In [ ]:
if __name__ == "__main__":
    main()